# Chapter 2: Working with Clinical Data in Pandas

## 1. Introduction

Now that we have our data loaded into DataFrames, a clinician at a medical center needs to start organizing it to uncover insights. In this section, we'll learn the essential skills for preparing data for analysis: sorting records to prioritize cases, merging disparate datasets to create a complete patient picture, and grouping data to compare outcomes. By mastering these techniques, you will learn how to structure datasets into a cohesive, analyzable format, a critical first step in tasks like identifying high-risk patients or evaluating treatment outcomes.

---

## 2. Key Concepts and Definitions

*   **Sorting**: The process of reordering rows in a DataFrame based on the values in one or more columns. In a clinical context, this is used to rank patients by a specific biomarker or to order events chronologically.
*   **Merging**: The action of combining two or more DataFrames based on a shared column, known as a "key." This is essential for creating a complete patient profile by linking demographic data with their corresponding lab results.
*   **Key (or Join Key)**: A column or set of columns containing common values that can be used to link rows between different DataFrames. A `patient_id` is a classic key used to merge various datasets related to a single individual.
*   **Grouping**: The process of splitting a DataFrame into groups based on some criteria (e.g., treatment arm, gender). This allows for calculations to be performed on each group independently.
*   **Aggregation**: The computation of a summary statistic (like a mean, count, or sum) for each group created by the `groupby()` method. For example, aggregating clinical trial data to calculate the average outcome for the placebo group versus the active drug group.
*   **Hemoglobin A1c (HbA1c)**: A blood test that measures average blood glucose levels over the past 3 months, commonly used to diagnose and monitor diabetes.
*   **Cholesterol**: A waxy substance found in the blood. High levels can increase the risk of heart disease. Measured in milligrams per deciliter (mg/dL).
*   **Blood Pressure (BP) Reduction**: The decrease in blood pressure, typically measured in millimeters of mercury (mmHg), often used as an endpoint in clinical trials for hypertension treatments.
*   **NaN (Not a Number)**: A special floating-point value that represents missing or undefined data in Pandas. It often appears after a merge operation when a row in one DataFrame does not have a corresponding match in the other.

---

## 3. Main Content

### 3.1 Sorting Data with `sort_values()`

The `sort_values()` method reorders a DataFrame. Use it to rank patients by lab results like Hemoglobin A1c (HbA1c) or sort appointments chronologically.

> **In Practice:** In a busy clinic, a nurse might need to quickly identify patients with the highest HbA1c levels for follow-up calls. Sorting the patient list in descending order by `HbA1c` instantly brings the most critical cases to the top, ensuring timely intervention.

In [ ]:
import pandas as pd

# Sample data of patient lab results
data = {'patient_id': ['PT000103', 'PT000101', 'PT000102', 'PT000104'],
        'age': [45, 34, 62, 50],
        'HbA1c': [6.8, 5.5, 7.1, 7.1]} # Hemoglobin A1c (%)
df = pd.DataFrame(data)

# Sort by age (ascending=True is the default)
sorted_by_age = df.sort_values(by='age')

# Sort by HbA1c (desc), then by age (asc)
sorted_multi = df.sort_values(by=['HbA1c', 'age'], ascending=[False, True])

print("--- Sorted by Age (Ascending) ---")
print(sorted_by_age)
print("\n--- Sorted by HbA1c (Desc), then Age (Asc) ---")
print(sorted_multi)

# --- Sorted by Age (Ascending) ---
#   patient_id  age  HbA1c
# 1   PT000101   34    5.5
# 0   PT000103   45    6.8
# 3   PT000104   50    7.1
# 2   PT000102   62    7.1
#
# --- Sorted by HbA1c (Desc), then Age (Asc) ---
#   patient_id  age  HbA1c
# 3   PT000104   50    7.1
# 2   PT000102   62    7.1
# 0   PT000103   45    6.8
# 1   PT000101   34    5.5

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




> **Pro Tip:** Always assign the result of `sort_values()` to a new variable (e.g., `sorted_df = df.sort_values(...)`). While using the `inplace=True` parameter might seem convenient, it modifies your original DataFrame directly, which can make your code harder to debug if you make a mistake.

### 3.2 Merging DataFrames with `pd.merge()`

Sorting helps us order data, but patient information is often spread across multiple files. Next, we'll learn how to combine these separate datasets into a single, unified DataFrame.

> **Medical Background:** A patient's data is rarely stored in one place. You'll often find demographic information in a hospital's registration system, lab results in a separate lab information system (LIS), and appointment history in a scheduling system. Merging is the digital equivalent of collating these different documents into a single, comprehensive patient chart before a consultation.

The `pd.merge()` function combines DataFrames on a shared key. This is essential for linking separate datasets, like patient demographics and their lab results.

In [ ]:
import pandas as pd

patients = pd.DataFrame({
    'patient_id': ['PT000101', 'PT000102', 'PT000103'],
    'name': ['John Doe', 'Jane Smith', 'Peter Jones']
})
appointments = pd.DataFrame({
    'patient_id': ['PT000101', 'PT000102'],
    'appt_date': ['2025-11-20', '2025-11-21']
})
vitals = pd.DataFrame({
    'patient_ref': ['PT000101', 'PT000102'], # Different key column name
    'heart_rate_bpm': [72, 88]
})

# Left merge: keeps all patients, filling missing appointments with NaN
merged_left = pd.merge(patients, appointments, on='patient_id', how='left')

# Inner merge: keeps only patients with appointments
merged_inner = pd.merge(patients, appointments, on='patient_id', how='inner')

# Merge with different key names
merged_keys = pd.merge(
    patients, vitals, left_on='patient_id', right_on='patient_ref'
)

print("--- Left Merge ---")
print(merged_left)
print("\n--- Inner Merge ---")
print(merged_inner)
print("\n--- Merge with Different Key Names ---")
print(merged_keys)

# --- Left Merge ---
#   patient_id         name   appt_date
# 0   PT000101     John Doe  2025-11-20
# 1   PT000102   Jane Smith  2025-11-21
# 2   PT000103  Peter Jones         NaN
#
# --- Inner Merge ---
#   patient_id        name   appt_date
# 0   PT000101    John Doe  2025-11-20
# 1   PT000102  Jane Smith  2025-11-21
#
# --- Merge with Different Key Names ---
#   patient_id         name patient_ref  heart_rate_bpm
# 0   PT000101     John Doe    PT000101              72
# 1   PT000102   Jane Smith    PT000102              88

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




> **Important:** Choosing the right `how` parameter is critical. A `left` merge on a master patient list ensures you don't accidentally drop patients from your analysis, even if they're missing data in the other table (e.g., no recent appointments). An `inner` merge is useful only when you need to analyze the intersection of two datasets, but be aware it will exclude any records that don't have a match.

### 3.3 Grouping and Aggregating Data with `groupby()`

With our data merged into a comprehensive view, we can now perform powerful summary analyses. Let's explore how to group data to compare different patient cohorts, a cornerstone of clinical trial analysis. The `groupby()` method splits data into groups to perform calculations. Use it with `.agg()` to calculate summary statistics, like the mean and count, for each treatment group in a clinical trial.

In [ ]:
import pandas as pd

trial_data = pd.DataFrame({
    'patient_id': ['PT000001', 'PT000002', 'PT000003', 'PT000004', 'PT000005'],
    'treatment_group': ['A', 'B', 'A', 'B', 'A'],
    'gender': ['M', 'F', 'F', 'F', 'M'],
    'bp_reduction': [10, 15, 12, 18, 9] # mmHg
})

# Group by multiple columns and calculate multiple statistics
group_stats = trial_data.groupby(['treatment_group', 'gender']).agg(
    mean_reduction=('bp_reduction', 'mean'),
    patient_count=('patient_id', 'count')
).reset_index()

print(group_stats)

#   treatment_group gender  mean_reduction  patient_count
# 0               A      F            12.0              1
# 1               A      M             9.5              2
# 2               B      F            16.5              2

**Try it yourself:** Modify the code above or write your own version

In [ ]:
# TODO: Write your code here
# Hint: Try modifying the example above




> **Debug Note:** A common mistake is forgetting to add `.reset_index()` after a `groupby().agg()` operation. Without it, the grouping columns (`treatment_group`, `gender`) become the DataFrame's index instead of regular columns, which can make subsequent data manipulation steps more complicated.

---

## 4. Practice Exercises

### Exercise 1: Sorting Lab Results Basic

**Objective:** Practice sorting a DataFrame by multiple columns with different sort orders.
**Time:** 5 minutes
**Medical Context:** A clinician needs to review patient cholesterol levels, prioritizing the highest readings first. For patients with the same cholesterol level, they want to see them in order of their ID.

You are given a DataFrame of lab results. Sort the `lab_results` DataFrame first by `cholesterol` in descending order, and then by `patient_id` in ascending order.

Sorting Lab Results Basic

**Objective:** Practice sorting a DataFrame by multiple columns with different sort orders.
**Time:** 5 minutes
**Medical Context:** A clinician needs to review patient cholesterol levels, prioritizing the highest readings first. For patients with the same cholesterol level, they want to see them in order of their ID.

You are given a DataFrame of lab results. Sort the `lab_results` DataFrame first by `cholesterol` in descending order, and then by `patient_id` in ascending order.

In [ ]:
import pandas as pd

lab_results = pd.DataFrame({
    'patient_id': ['PT000201', 'PT000202', 'PT000203', 'PT000204'],
    'cholesterol': [220, 195, 220, 210] # mg/dL
})

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
# Solution
sorted_labs = lab_results.sort_values(
    by=['cholesterol', 'patient_id'], 
    ascending=[False, True]
)
print(sorted_labs)

#    patient_id  cholesterol
# 0    PT000201          220
# 2    PT000203          220
# 3    PT000204          210
# 1    PT000202          195
```
**Explanation:** This code first sorts all patients by `cholesterol` from highest to lowest (`ascending=False`). For the two patients with the same cholesterol level (220), it then applies the second sorting rule, ordering them by `patient_id` from lowest to highest (`ascending=True`). This two-level sorting is crucial for creating reproducible and consistently ordered reports.
**Key Learning:** Sorting by multiple criteria is a common task for prioritizing data, such as identifying high-risk patients.


</div>
</details>

### Exercise 2: Merging Patient Rosters Intermediate

**Objective:** Perform a merge that retains all records from a primary DataFrame.
**Time:** 5 minutes
**Medical Context:** A hospital administrator needs a complete list of all registered patients and their scheduled appointment dates. It is critical that no patient is dropped from the list, even if they don't have an upcoming appointment.

You have `patient_registry` and `appointments`. Perform a merge that includes all patients from `patient_registry` and adds their appointment date if one exists.

Merging Patient Rosters Intermediate

**Objective:** Perform a merge that retains all records from a primary DataFrame.
**Time:** 5 minutes
**Medical Context:** A hospital administrator needs a complete list of all registered patients and their scheduled appointment dates. It is critical that no patient is dropped from the list, even if they don't have an upcoming appointment.

You have `patient_registry` and `appointments`. Perform a merge that includes all patients from `patient_registry` and adds their appointment date if one exists.

In [ ]:
import pandas as pd

patient_registry = pd.DataFrame({
    'patient_id': ['PT000301', 'PT000302', 'PT000303'],
    'name': ['Alice', 'Bob', 'Charlie']
})
appointments = pd.DataFrame({
    'patient_id': ['PT000301', 'PT000303'],
    'appt_date': ['2025-12-01', '2025-12-02']
})

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
# Solution
all_patients_appts = pd.merge(
    patient_registry, 
    appointments, 
    on='patient_id', 
    how='left'
)
print(all_patients_appts)

#   patient_id     name   appt_date
# 0   PT000301    Alice  2025-12-01
# 1   PT000302      Bob         NaN
# 2   PT000303  Charlie  2025-12-02
```
**Explanation:** The request was to include *all* patients from the `patient_registry`. A `left` merge accomplishes this by using `patient_registry` as the "left" DataFrame and keeping every row from it. Notice that patient 'Bob' (ID `PT000302`), who does not have an appointment, is still in the final DataFrame, but his `appt_date` is `NaN` (Not a Number), indicating missing data.
**Key Learning:** The `how='left'` merge is crucial when you need to enrich a primary dataset without losing any of its original records.


</div>
</details>

### Exercise 3: Summarizing Trial Data Intermediate

**Objective:** Practice using `groupby()` with `.agg()` to calculate multiple summary statistics.
**Time:** 5 minutes
**Medical Context:** A data scientist is analyzing results from a small clinical trial. They need to calculate the number of patients and the average blood pressure reduction for each treatment group to prepare a preliminary efficacy report.

Using the `trial_data` DataFrame from the lesson, group the data by `treatment_group` only. Use a single `.agg()` call to find the number of patients (`patient_count`) and the average blood pressure reduction (`avg_reduction`).

Summarizing Trial Data Intermediate

**Objective:** Practice using `groupby()` with `.agg()` to calculate multiple summary statistics.
**Time:** 5 minutes
**Medical Context:** A data scientist is analyzing results from a small clinical trial. They need to calculate the number of patients and the average blood pressure reduction for each treatment group to prepare a preliminary efficacy report.

Using the `trial_data` DataFrame from the lesson, group the data by `treatment_group` only. Use a single `.agg()` call to find the number of patients (`patient_count`) and the average blood pressure reduction (`avg_reduction`).

In [ ]:
import pandas as pd

trial_data = pd.DataFrame({
    'patient_id': ['PT000001', 'PT000002', 'PT000003', 'PT000004', 'PT000005'],
    'treatment_group': ['A', 'B', 'A', 'B', 'A'],
    'gender': ['M', 'F', 'F', 'F', 'M'],
    'bp_reduction': [10, 15, 12, 18, 9] # mmHg
})

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```python
# Solution
summary_stats = trial_data.groupby('treatment_group').agg(
    patient_count=('patient_id', 'count'),
    avg_reduction=('bp_reduction', 'mean')
).reset_index()
print(summary_stats)

#   treatment_group  patient_count  avg_reduction
# 0               A              3      10.333333
# 1               B              2      16.500000
```
**Explanation:** This single command efficiently calculates multiple summary statistics for each subgroup. The named aggregation—`patient_count=('patient_id', 'count')`—makes the resulting DataFrame immediately understandable. The output clearly shows the number of patients and the average blood pressure reduction for treatment groups 'A' and 'B', providing a powerful snapshot of the trial's results.
**Key Learning:** The `groupby().agg()` pattern is the standard and most readable way to generate summary statistics for different subgroups in your data.


</div>
</details>

---

## 5. Practical Applications

*   **Identifying High-Risk Patient Cohorts**: A hospital can merge patient electronic health records (EHR) with recent lab results using `pd.merge()`. They can then use `sort_values()` on columns like `HbA1c` or `LDL_cholesterol` to rank patients and identify those at highest risk for diabetes or cardiovascular events, enabling proactive outreach.
*   **Evaluating Clinical Trial Efficacy**: Researchers use `groupby('treatment_group')` and `.agg()` to analyze clinical trial data. By calculating the mean, median, and standard deviation of outcome measures (like tumor size reduction or blood pressure change) for both the drug and placebo groups, they can rigorously assess a new therapy's effectiveness.
*   **Resource Allocation and Planning**: A public health agency can merge demographic data with disease prevalence data from different regions. By grouping the combined data by `region` and `age_group` using `groupby()`, they can calculate incidence rates and identify hotspots, which informs where to allocate testing facilities, vaccines, or educational campaigns.
*   **Creating a Longitudinal Patient View**: To study disease progression, analysts merge multiple datasets containing patient visits over time using `pd.merge()` on `patient_id`. They then use `sort_values()` on `visit_date` to create a chronological history for each patient, which is essential for longitudinal analysis and modeling.

> **Reflection Moment:** In our example, we grouped by treatment arm and gender. What other variables in a clinical trial dataset (e.g., age group, pre-existing conditions, clinic location) might be useful for a `groupby` analysis? How could these deeper analyses help researchers understand a new drug's effectiveness more thoroughly?

---

## 6. Summary and Key Takeaways

In this section, we've explored the three core pillars of data manipulation in Pandas. We learned how to bring order to our data, combine it from different sources, and calculate meaningful summary statistics for different subgroups. These skills are not just technical exercises; they are the foundational activities for nearly all data analysis in precision health.

*   **`sort_values()`** is used to reorder data based on column values, which is critical for ranking and prioritization.
*   **`pd.merge()`** is the essential tool for combining disparate datasets into a single, unified DataFrame using a common key.
*   **`groupby().agg()`** is the standard pattern for splitting data into groups, applying calculations, and

---